# Ejercicio 3

Implemente un perceptrón multicapa que aprenda la función lógica XOR de 2 y de 4 entradas (utilizando el algoritmo Backpropagation y actualizando en batch). Muestre cómo evoluciona el error durante el entrenamiento.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(110007)

In [ ]:
class MLP:
    # layers incluye la entrada, por ejemplo (4, 8, 2) son 4 entradas,
    # una capa oculta con 8 neuronas, y 2 neuronas de salida
    def __init__(self, layers):
        self.weights = []
        self.n_layers = len(layers) - 1
        for (n_neurons, n_inputs) in zip(layers[1:], layers):
            self.weights.append(np.random.normal(0, 1.0, size=(n_neurons, n_inputs + 1)))

    def predict(self, X):
        n_p = X.shape[0]
        y = []
        for k in range(n_p):
            Vk = self.forward(X[k])
            y.append(Vk[-1])
        return np.array(y)

    def forward(self, X):
        V = [X]
        for m in range(self.n_layers):
            X_ext = np.append(V[m], 1.0)
            Vm = self.activation(np.dot(self.weights[m], X_ext))
            V.append(Vm)
        return V

    def train(self, X, y, epochs=1000, lr=1e-3):
        n_p = X.shape[0]

        self.losses = []
        for _ in range(epochs):
            for k in np.random.permutation(n_p):
                Xk = X[k]
                yk = y[k]
                Vk = self.forward(Xk)
                delta = [None] * self.n_layers
                delta[-1] = (yk - Vk[-1]) * self.activation_dif(Vk[-1])
                for m in range(self.n_layers-1, 0, -1):
                    delta[m-1] = (self.weights[m][:, :-1].T @ delta[m]) * self.activation_dif(Vk[m])
                for m in range(self.n_layers):
                    V_ext = np.append(Vk[m], 1.0)
                    self.weights[m] += lr * np.outer(delta[m], V_ext)
            self.losses.append(self.error(X, y))

    def error(self, X, y):
        y_pred = self.predict(X).reshape(y.shape)
        return 0.5 * np.sum((y_pred - y) ** 2)

    @staticmethod
    def activation(h):
        return 1 / (1 + np.exp(-h))

    @staticmethod
    def activation_dif(V):
        return V * (1 - V)

In [ ]:
X2 = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
y2_xor  = np.array([0, 1, 1, 0])

In [ ]:
model = MLP([2, 2, 1])

In [ ]:
model.train(X2, y2_xor, lr=0.2, epochs=5000)
for w in model.weights:
    print(w)

In [ ]:
y_pred = model.predict(X2)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(model.losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(4, 4))

x1_min, x1_max = -0.5, 1.51
x2_min, x2_max = -0.5, 1.51
    
xx1, xx2 = np.meshgrid(np.arange(x1_min, x2_max, 0.01),
                     np.arange(x2_min, x2_max, 0.01))
X = np.c_[xx1.ravel(), xx2.ravel()]
y = model.predict(X).flatten().reshape(xx1.shape)
plt.contourf(xx1, xx2, y, levels=[0, 0.5, 1], alpha=0.4, cmap='bwr')
plt.contour(xx1, xx2, y, levels=[0.5], colors='k')
plt.scatter(X2[:, 0], X2[:, 1], c=y_pred, cmap='bwr', s=200, edgecolors='k')
plt.xlabel('Entrada $X_1$')
plt.ylabel('Entrada $X_2$')
plt.xlim([-0.5, 1.5])
plt.ylim([-0.5, 1.5])
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
X4 = np.array(list(itertools.product([0, 1], repeat=4)))
y4_xor  = np.where(np.sum(X4, axis=1) % 2 == 1, 1, 0)